# Retail shoplifting — Experiment B GPU training

Test whether explicit joint motion helps compared with Experiment A. Use normalized 2D positions plus per-frame displacements of shoulders, elbows, wrists, hips, knees and ankles: **58 features per frame**. This is the existing BiGRU Autoencoder with the same hidden size and depth; its input/output parameter count grows with the added features.

The package preserves A’s 100-file sample, 35,557 training windows, 6,284 normal validation windows and 1,856 retained test windows. Only the feature family changes. No new videos or pose extraction are needed.

Upload `Retail_Shoplifting_Colab_Training_B.zip` to **My Drive → Shoplifting**. Select a **GPU runtime** (T4 when available), then run cells one at a time. Keep A’s existing files. B outputs use a separate directory.

This exploratory experiment follows inspection of test results; it cannot establish generalization without fresh confirmation data. No guarantee of improvement is made.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ZIP = Path("/content/drive/MyDrive/Shoplifting/Retail_Shoplifting_Colab_Training_B.zip")
OUT = Path("/content/drive/MyDrive/Shoplifting/Retail_Shoplifting_Training_B_v1")
ROOT = Path("/content/retail_training_b")
assert ZIP.is_file(), f"Upload the training ZIP to My Drive: {ZIP}"

## 1. Extract and verify inputs
Check: verification reports 41,841 normal windows, a disjoint source-video split, and 1,856 retained test windows. The ZIP contains prepared pose tensors, not RGB videos. It preserves the current RetailS-specific coordinate interpretation and quality gates; this does not establish pose or detection accuracy.

In [ ]:
import zipfile, hashlib, json
with zipfile.ZipFile(ZIP) as archive:
    for entry in archive.infolist():
        target = (ROOT / entry.filename).resolve()
        assert target.is_relative_to(ROOT.resolve()), "Unsafe archive path"
    archive.extractall(ROOT)
manifest = json.loads((ROOT / "bundle_manifest.json").read_text())
for name, expected in manifest["files"].items():
    with (ROOT / name).open("rb") as f:
        assert hashlib.file_digest(f, "sha256").hexdigest() == expected, name
print("Integrity check passed")
print(json.dumps({"split": manifest["split"], "counts": {k: v["counts"] for k,v in manifest["datasets"].items()}}, indent=2))

## 2. Check the GPU and dependencies
Check: CUDA must be available. Keep Colab’s installed PyTorch build; the notebook only installs missing supporting packages.

In [ ]:
import importlib.util, subprocess, sys
for module, package in [("numpy", "numpy"), ("sklearn", "scikit-learn"), ("matplotlib", "matplotlib"), ("pandas", "pandas")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
import torch
assert torch.cuda.is_available(), "Select a GPU runtime, reconnect, then rerun these cells"
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))

## 3. Train Experiment B — three seeds

Run fresh B models with 58 features per frame: 34 normalized positions and 24 joint displacements. Keep 15 epochs, batch size 32, Adam learning rate 0.001, hidden size 128, two layers and dropout 0.2. Seeds are 0/1/2. Training/validation windows and labels match A exactly.

Check each seed reaches 15/15. Epoch checkpoints save to the separate B output directory on Drive. Rerun this cell after reconnecting to resume saved epochs. Do not point B to the A output directory. Best epochs and p95 thresholds use normal validation only. Loss scales differ because B includes additional features; compare detection metrics rather than raw A/B loss values.

In [ ]:
subprocess.run([sys.executable, "-u", str(ROOT / "scripts/train_colab.py"), "--data-root", str(ROOT), "--output", str(OUT), "--device", "cuda", "--seeds", "0", "1", "2"], check=True)

## 4. Verify and inspect results
Check: three completed runs, 15 logged epochs per run, finite metrics. Precision alone can be misleading because the retained test windows contain many anomalies. Report recall, F1, ROC-AUC, PR-AUC and rejected-window coverage as well. These are video-mask-derived anomaly labels, not person-level theft annotations.

In [ ]:
import pandas as pd
rows = []
for seed in (0, 1, 2):
    folder = OUT / f"expanded_b_seed{seed}"
    complete = json.loads((folder / "complete.json").read_text())
    for name, expected in complete["files"].items():
        with (folder / name).open("rb") as f:
            assert hashlib.file_digest(f, "sha256").hexdigest() == expected, name
    assert len(json.loads((folder / "history.json").read_text())) == 15
    report = json.loads((folder / "evaluation.json").read_text())
    rows.append({"seed": seed, "best_epoch": report["best_epoch"], **report["metrics"]["combined"]})
table = pd.DataFrame(rows)
display(table)
display(table[["precision", "recall", "f1", "roc_auc", "pr_auc"]].agg(["mean", "std"]))
assert all(json.loads((OUT / f"expanded_b_seed{s}" / "evaluation.json").read_text())["experiment"] == "b" for s in (0,1,2))
baseline = json.loads((ROOT / "baseline_a.json").read_text())
a = pd.DataFrame(baseline["rows"])
print("A versus B on identical retained test windows (means across three seeds):")
display(pd.DataFrame({"A_positions": a[["precision","recall","f1","roc_auc","pr_auc"]].mean(), "B_positions_velocity": table[["precision","recall","f1","roc_auc","pr_auc"]].mean()}))
print("This is an exploratory follow-up to inspected test results; improvement needs fresh confirmation data.")

## 5. Export results for the local repository
Download the results ZIP, place it in the local repository’s `outputs/colab_handoff/` directory, and follow `COLAB_EXPERIMENT_B.md`. Keep Drive checkpoints until local validation is complete. The export includes the CPU-loadable best weights, scores, normal calibration errors, logs, provenance and checksums. Full resume states remain on Drive.

In [ ]:
RESULT = OUT.parent / "Retail_Shoplifting_Colab_Results_B.zip"
with zipfile.ZipFile(RESULT, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUT / "bundle_manifest.json", "bundle_manifest.json")
    for subset in ("train", "test_realworld", "test_staged"):
        archive.write(ROOT / f"data/{subset}.json", f"data/{subset}.json")
    for seed in (0, 1, 2):
        folder = OUT / f"expanded_b_seed{seed}"
        complete = json.loads((folder / "complete.json").read_text())
        for name in ["complete.json", *complete["files"]]:
            archive.write(folder / name, f"{folder.name}/{name}")
print(RESULT, round(RESULT.stat().st_size / 1024**2, 1), "MiB")
from google.colab import files
files.download(str(RESULT))